In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()


In [ ]:
!pip install --upgrade pinecone-client pinecone-text pinecone-notebooks
!pip install -U pinecone

  Using cached pinecone_client-6.0.0-py3-none-any.whl.metadata (3.4 kB)
  Using cached pinecone_text-0.11.0-py3-none-any.whl.metadata (10 kB)

  Attempting uninstall: mmh3

    Found existing installation: mmh3 5.2.0

    Uninstalling mmh3-5.2.0:

   ---------------------------------------- 0/6 [mmh3]
   ---------------------------------------- 0/6 [mmh3]
   ---------------------------------------- 0/6 [mmh3]
   ---------------------------------------- 0/6 [mmh3]
   ---------------------------------------- 0/6 [mmh3]
   ---------------------------------------- 0/6 [mmh3]
   ---------------------------------------- 0/6 [mmh3]
   ---------------------------------------- 0/6 [mmh3]
   ---------------------------------------- 0/6 [mmh3]
   ---------------------------------------- 0/6 [mmh3]
   ---------------------------------------- 0/6 [mmh3]
   ---------------------------------------- 0/6 [mmh3]
   ---------------------------------------- 0/6 [mmh3]
   ----------------------------------

In [4]:
import  os 
from dotenv import load_dotenv
load_dotenv()

api_key = os.getenv("PINECONE_API_KEY")


In [9]:
from langchain_community.retrievers import PineconeHybridSearchRetriever
from pinecone import Pinecone, ServerlessSpec

# Initialize Pinecone

index_name = "langchain-retriever"

pc = Pinecone(api_key=api_key)


if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=1536,
        metric="cosine",
        serverless=ServerlessSpec(cloud="aws", region="us-east-1"),
        
    )  

In [11]:
print(f"Index '{index_name}' is ready to use.")
index = pc.Index(index_name)
index

Index 'langchain-retriever' is ready to use.


In [22]:
import os
import certifi

os.environ["SSL_CERT_FILE"] = certifi.where()

In [25]:
from langchain_community.embeddings import HuggingFaceBgeEmbeddings

embedding = HuggingFaceBgeEmbeddings(
    model_name="all-MiniLM-L6-v2"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 470.21it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [26]:
embedding

HuggingFaceBgeEmbeddings(client=SentenceTransformer(
  (0): Transformer({'max_seq_length': 256, 'do_lower_case': False, 'architecture': 'BertModel'})
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
), model_name='all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_instruction='Represent this question for searching relevant passages: ', embed_instruction='', show_progress=False)

In [27]:
from pinecone_text.sparse import BM25Encoder

sparse_encoder = BM25Encoder().default()
sparse_encoder

In [28]:
sentences = [
    "The cat is on the table.",
    "The dog is in the garden.",
    "The bird is flying in the sky."
]
sparse_encoder.fit(sentences)
sparse_encoder.dump("sparse_encoder.json")
sparse_encoder=BM25Encoder().load("sparse_encoder.json")
sparse_encoder

100%|██████████| 3/3 [00:00<00:00, 57.11it/s]


In [30]:
retriever = PineconeHybridSearchRetriever(
    index=index,
    embeddings=embedding,
    sparse_encoder=sparse_encoder,
)

In [32]:
retriever.add_texts(
    [
    "The cat is on the table.",
    "The dog is in the garden.",
    "The bird is flying in the sky."
]
)

100%|██████████| 1/1 [00:03<00:00,  3.17s/it]


In [34]:
retriever.invoke("who is on the table?")

[Document(metadata={'score': 0.34987694}, page_content='The cat is on the table.'),
 Document(metadata={'score': -0.00844335556}, page_content='The bird is flying in the sky.'),
 Document(metadata={'score': -0.02175951}, page_content='The dog is in the garden.')]

In [39]:
retriever.invoke("who is in the garden?",top_k=2)

TypeError: pinecone.db_data.index.Index.query() got multiple values for keyword argument 'top_k'